In [25]:
import pandas as pd
import numpy as np
import random

df_scores = pd.read_parquet('../temp/df_scores.parquet')
s_weights = pd.read_parquet('../temp/feature_weights.parquet')[0]
df_scores.columns = s_weights.index
df_allocation = pd.read_parquet('../temp/df_allocation.parquet')

df_allocation = df_allocation.loc[df_allocation.sum(1) >0]
s_scores = pd.Series(np.matmul(df_scores.values, s_weights.values.reshape(len(s_weights),1)).flatten(), index = df_scores.index)
random_tickers = random.sample(list(set(df_scores.index).difference(df_allocation.index)), 10)

df_best = df_scores.loc[df_allocation.index]
df_random = df_scores.loc[random_tickers]

best_scores = s_scores.loc[df_best.index]
random_scores = s_scores.loc[df_random.index]



In [40]:
df_best


,dollar_ret_1p,dollar_ret_6p,dollar_ret_13p,dollar_ret_26p,avg_eps_1q,avg_eps_2q,avg_eps_4q,avg_eps_8q
AES,0.137354,-0.149505,0.208415,-0.009975,0.208814,0.208814,0.208814,0.212203
AMD,-0.802571,5.545575,1.707231,0.843266,0.011496,0.011496,0.011496,0.011822
BKNG,17.257723,1.580049,0.000799,0.395534,0.033246,0.033246,0.033246,0.564815
CAG,0.080139,-0.040503,-0.019294,-0.027609,0.014169,0.012964,0.012361,0.012737
CCL,0.314184,-0.059368,-0.005367,0.144591,0.021858,0.016260,0.013461,0.014861
CNC,-0.275527,1.500578,1.600667,-0.060600,0.197220,0.197220,0.197220,0.076152
CRL,30.285913,6.210356,2.501369,0.563821,0.086211,0.086211,0.086211,0.093325
CRWD,8.543971,6.055951,0.997287,0.842739,0.004444,0.004444,0.004444,0.004285
CVS,-0.919030,0.846353,0.675494,0.466378,0.138750,0.138750,0.138750,0.102241
DDOG,0.051460,12.375070,1.931695,0.989361,0.017291,0.017291,0.017291,0.017003


In [42]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.express as px

def create_feature_rectangle_plot(df_best, df_random, s_weights):
    features = list(df_best.columns)
    base_weights = s_weights.loc[features].values.astype(float)
    total_width = np.sum(base_weights)

    # Combine datasets with a source label
    df_best_plot = df_best.copy()
    df_best_plot['Group'] = 'Best Stocks'
    df_random_plot = df_random.copy()
    df_random_plot['Group'] = 'Random Stocks'
    
    combined = pd.concat([df_best_plot, df_random_plot]).reset_index(names=['Ticker'])
    
    # Calculate total score (sum of weight * value products) for each row
    scores = []
    global_min_y = float('inf')
    global_max_y = float('-inf')
    
    for row in combined.itertuples():
        values = np.array([getattr(row, f) for f in features], dtype=float)
        score = np.sum(base_weights * values)
        scores.append(score)
        
        global_min_y = min(global_min_y, np.min(values))
        global_max_y = max(global_max_y, np.max(values))
    
    combined['Total_Score'] = scores
    
    # Split into two groups and sort each by total score descending
    df_best_sorted = combined[combined['Group'] == 'Best Stocks'].sort_values(by='Total_Score', ascending=False).reset_index(drop=True)
    df_random_sorted = combined[combined['Group'] == 'Random Stocks'].sort_values(by='Total_Score', ascending=False).reset_index(drop=True)
    
    best_rows = len(df_best_sorted)
    random_rows = len(df_random_sorted)
    total_rows = best_rows + random_rows

    v_spacing = min(0.01, 0.3 / max(total_rows - 1, 1))

    # Construct subplot titles
    subplot_titles = []
    for i in range(best_rows):
        row = df_best_sorted.iloc[i]
        if i == 0:
            subplot_titles.append(f"<b>BEST STOCKS</b><br>{row.Ticker} — Score: {row.Total_Score:.2f}")
        else:
            subplot_titles.append(f"{row.Ticker} — Score: {row.Total_Score:.2f}")
            
    for i in range(random_rows):
        row = df_random_sorted.iloc[i]
        if i == 0:
            subplot_titles.append(f"<b>RANDOM STOCKS</b><br>{row.Ticker} — Score: {row.Total_Score:.2f}")
        else:
            subplot_titles.append(f"{row.Ticker} — Score: {row.Total_Score:.2f}")

    fig = make_subplots(
        rows=total_rows, cols=1,
        subplot_titles=subplot_titles,
        vertical_spacing=v_spacing
    )

    # Consistent color mapping for each feature, with manual override for 'eps_4q'
    color_palette = px.colors.qualitative.Plotly
    feature_colors = {}
    for i, feat in enumerate(features):
        if feat == 'eps_4q':
            feature_colors[feat] = '#FFD700' # Yellow
        else:
            feature_colors[feat] = color_palette[i % len(color_palette)]

    combined_sorted = pd.concat([df_best_sorted, df_random_sorted]).reset_index(drop=True)

    for i, row in enumerate(combined_sorted.itertuples(), start=1):
        values = np.array([getattr(row, f) for f in features], dtype=float)
        
        # Calculate contribution (weight * value) and sort descending per row
        contributions = base_weights * values
        sorted_indices = np.argsort(-contributions)
        
        row_features = [features[idx] for idx in sorted_indices]
        row_weights = base_weights[sorted_indices]
        row_values = values[sorted_indices]
        row_contributions = contributions[sorted_indices]
        
        # Calculate dynamic midpoints and widths for this row's sorted order
        cum_weights = np.concatenate(([0], np.cumsum(row_weights)))
        midpoints = (cum_weights[:-1] + cum_weights[1:]) / 2

        for j, feat in enumerate(row_features):
            c = feature_colors[feat]
            w = row_weights[j]
            v = row_values[j]
            contrib = row_contributions[j]
            
            # Custom hover template keeping text hidden until mouseover
            custom_hover = (
                f"<b>Feature:</b> {feat}<br>"
                f"<b>Weight:</b> {w:.4f}<br>"
                f"<b>Value:</b> {v:.4f}<br>"
                f"<b>Contribution:</b> {contrib:.4f}<extra></extra>"
            )

            fig.add_trace(
                go.Bar(
                    x=[midpoints[j]],
                    y=[v],
                    width=[w],
                    marker=dict(color=c),
                    name=feat,
                    legendgroup=feat,
                    showlegend=(i == 1),
                    hovertemplate=custom_hover
                ),
                row=i, col=1
            )

        fig.add_hline(y=0, line_width=1, line_dash="dash", line_color="gray", row=i, col=1)

    fig.update_layout(
        height=max(900, 220 * total_rows),
        title_text="Stock Feature Profiles (Width = Feature Weight, Height = Value)",
        barmode='relative',
        bargap=0
    )
    
    y_padding = max(abs(global_max_y), abs(global_min_y)) * 0.1
    y_range = [min(0, global_min_y) - y_padding, max(0, global_max_y) + y_padding]

    for i in range(1, total_rows + 1):
        fig.update_xaxes(range=[0, total_width], showticklabels=False, row=i, col=1)
        fig.update_yaxes(range=y_range, row=i, col=1)

    return fig

fig = create_feature_rectangle_plot(df_best, df_random, s_weights)
fig.show()